In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
data = pd.read_csv('Dry_Bean_Dataset.csv', delimiter=';', decimal=',')
print(data.columns.tolist())
data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)
X = data.drop(['Class'], axis=1).values
le = LabelEncoder()
y = le.fit_transform(data['Class'].values)
#y = data['Class'].values
#y_decoded = le.inverse_transform(y_encoded)  

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

print(pd.Series(y).value_counts())



scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
Признаков: 16, Объектов: 13611
3    3546
6    2636
5    2027
4    1928
2    1630
0    1322
1     522
Name: count, dtype: int64


In [25]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [26]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred,average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred)
print(accuracy, precision, recall, f1)
print(report)

0.9261843554902681 0.9281215893254025 0.9261843554902681 0.926537247992984
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       261
           1       1.00      1.00      1.00       117
           2       0.95      0.94      0.94       317
           3       0.94      0.88      0.91       671
           4       0.97      0.97      0.97       408
           5       0.96      0.94      0.95       413
           6       0.84      0.91      0.87       536

    accuracy                           0.93      2723
   macro avg       0.94      0.94      0.94      2723
weighted avg       0.93      0.93      0.93      2723



In [27]:
svc_model = SVC(
        C=100,
        kernel='rbf',
        gamma='scale',
        random_state=42)

svc_model.fit(X_train, y_train)
svc_pred = svc_model.predict(X_test)

svc_acc = accuracy_score(y_test, svc_pred)
svc_precision = precision_score(y_test, svc_pred, average='weighted')
svc_recall = recall_score(y_test, svc_pred, average='weighted')
svc_f1 = f1_score(y_test, svc_pred, average='weighted')
report = classification_report(y_test, svc_pred)
print(report)

              precision    recall  f1-score   support

           0       0.92      0.92      0.92       261
           1       1.00      1.00      1.00       117
           2       0.94      0.94      0.94       317
           3       0.90      0.93      0.92       671
           4       0.97      0.95      0.96       408
           5       0.97      0.96      0.96       413
           6       0.88      0.88      0.88       536

    accuracy                           0.93      2723
   macro avg       0.94      0.94      0.94      2723
weighted avg       0.93      0.93      0.93      2723



In [9]:

data = pd.read_csv('Dry_Bean_Dataset.csv', delimiter=';', decimal=',')
print(data.columns.tolist())

data = data.dropna(axis=1, how='all')  
data.fillna(data.mean(numeric_only=True), inplace=True)

X = data.drop(['Class'], axis=1).values
le = LabelEncoder()
y = le.fit_transform(data['Class'].values)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")
print(pd.Series(y).value_counts())

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)


class BeanClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super(BeanClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

model = BeanClassifier(X.shape[1], len(np.unique(y)))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(30):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Эпоха {epoch+1}/30, Потери: {running_loss/len(train_loader):.4f}")


model.eval()
y_pred = []
y_true = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_pred.extend(predicted.numpy())
        y_true.extend(labels.numpy())


print(f"Точность: {np.mean(np.array(y_pred) == np.array(y_true)):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=le.classes_))

['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
Признаков: 16, Объектов: 13611
3    3546
6    2636
5    2027
4    1928
2    1630
0    1322
1     522
Name: count, dtype: int64
Эпоха 10/30, Потери: 0.1924
Эпоха 20/30, Потери: 0.1810
Эпоха 30/30, Потери: 0.1722
Точность: 0.9266

Classification Report:
              precision    recall  f1-score   support

    BARBUNYA       0.91      0.93      0.92       261
      BOMBAY       1.00      1.00      1.00       117
        CALI       0.93      0.94      0.93       317
    DERMASON       0.89      0.94      0.92       671
       HOROZ       0.98      0.92      0.95       408
       SEKER       0.98      0.93      0.95       413
        SIRA       0.89      0.88      0.88       536

    accuracy                           0.93      2723
   macro avg  